# Open Measures Script
Collection, sampling, and cleaning of Open Measures data, collected from 4chan, 8kun, Bluesky and Truthsocial across 2025, stratified by month and query.

## Setup

In [7]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
API_URL = "https://pro.api.openmeasures.io"
jwt_token = os.getenv("OPEN_MEASURES_TOKEN")

_headers = {
    "Authorization": f"Bearer {jwt_token}"
}

In [8]:
# see quota
response = requests.get(f"{API_URL}/quota", headers=_headers)
data = response.json()

print(f"Status: {response.status_code}")
print(f"{data['organization_usage']['core_api_monthly_requests_count'] / data['monthly_limits']['core_api_monthly_request_limit'] * 100:.2f}% of monthly quota used")

Status: 200
6.61% of monthly quota used


## Sample Statistics

In [4]:
# Check and compare query sizes
from query import ANTI, ISRAEL, PALESTINE

a = ANTI
i = ISRAEL
p = PALESTINE

# dedupe is after sampling for fuzzy-identical cases
queries = {
    "ip_not": f'(({i}) OR ({p})) AND NOT ({a})',
    "ip_and": f'(({i}) OR ({p})) AND ({a})',
    "anti_only": f'({a}) AND NOT (({i}) OR ({p}))',
}

sites_to_test = ["4chan", "8kun", "truthsocial", "bluesky"]

results_summary = {}

for query_label, term_query in queries.items():
    results_summary[query_label] = {}
    for site in sites_to_test:
        test_params = {
            "sortdesc": "true",
            "limit": 1,
            "site": site,
            "term": term_query,
            "since": "2025-01-01",
            "until": "2025-12-31",
            "standard_fields": "true",
            "querytype": "boolean_content",
        }
        response = requests.get(f"{API_URL}/content", headers=_headers, params=test_params)
        print(f"====| {query_label} | {site} |====")
        print(f"Status: {response.status_code}")
        if response.status_code == 200:
            data = response.json()
            total_hits = data.get('total_hits')
            print(f"Total hits: {total_hits}")
            results_summary[query_label][site] = total_hits
        else:
            print(response.text[:300])
            results_summary[query_label][site] = None
        print()

print("\n=== SUMMARY ===")
for query_label, site_hits in results_summary.items():
    print(f"\n{query_label}:")
    for site, hits in site_hits.items():
        print(f"  {site}: {hits}")

====| ip_not | 4chan |====
Status: 200
Total hits: 887403

====| ip_not | 8kun |====
Status: 200
Total hits: 38070

====| ip_not | truthsocial |====
Status: 200
Total hits: 1148190

====| ip_not | bluesky |====
Status: 200
Total hits: 9783458

====| ip_and | 4chan |====
Status: 200
Total hits: 308888

====| ip_and | 8kun |====
Status: 200
Total hits: 9458

====| ip_and | truthsocial |====
Status: 200
Total hits: 132377

====| ip_and | bluesky |====
Status: 200
Total hits: 406729

====| anti_only | 4chan |====
Status: 200
Total hits: 2906601

====| anti_only | 8kun |====
Status: 200
Total hits: 30317

====| anti_only | truthsocial |====
Status: 200
Total hits: 576697

====| anti_only | bluesky |====
Status: 200
Total hits: 1376242


=== SUMMARY ===

ip_not:
  4chan: 887403
  8kun: 38070
  truthsocial: 1148190
  bluesky: 9783458

ip_and:
  4chan: 308888
  8kun: 9458
  truthsocial: 132377
  bluesky: 406729

anti_only:
  4chan: 2906601
  8kun: 30317
  truthsocial: 576697
  bluesky: 1376242

In [5]:
# Descriptive statistics

import pandas as pd

strata = ["ip_not", "ip_and", "anti_only"]

# results_summary structure: {query_label: {site: total_hits}}
df = pd.DataFrame(results_summary).T  # rows = query variant, columns = site
df.index.name = "query_variant"

# reconstruct 'full' as the sum of the three mutually exclusive partitions (sanity check + denominator)
df.loc["full"] = df.loc[strata].sum()

print("=== Raw hit counts per stratum ===")
print(df.to_string())

print("\n=== Composition: % of full corpus each stratum represents ===")
composition_df = (df.div(df.loc["full"], axis=1) * 100).round(2)
print(composition_df.to_string())

print("\n=== Composition by site ===")
for site in df.columns:
    print(f"\n{site}: total = {df.loc['full', site]:,}")
    for stratum in strata:
        pct = composition_df.loc[stratum, site]
        count = df.loc[stratum, site]
        print(f"  {stratum:<12} {count:>10,} ({pct:>5.2f}%)")

print("\n=== Cross-platform share: where does each stratum's volume come from? ===")
row_share_df = (df.div(df.sum(axis=1), axis=0) * 100).round(2)
print(row_share_df.loc[strata].to_string())

=== Raw hit counts per stratum ===
                 4chan   8kun  truthsocial   bluesky
query_variant                                       
ip_not          887403  38070      1148190   9783458
ip_and          308888   9458       132377    406729
anti_only      2906601  30317       576697   1376242
full           4102892  77845      1857264  11566429

=== Composition: % of full corpus each stratum represents ===
                4chan    8kun  truthsocial  bluesky
query_variant                                      
ip_not          21.63   48.90        61.82    84.58
ip_and           7.53   12.15         7.13     3.52
anti_only       70.84   38.95        31.05    11.90
full           100.00  100.00       100.00   100.00

=== Composition by site ===

4chan: total = 4,102,892
  ip_not          887,403 (21.63%)
  ip_and          308,888 ( 7.53%)
  anti_only     2,906,601 (70.84%)

8kun: total = 77,845
  ip_not           38,070 (48.90%)
  ip_and            9,458 (12.15%)
  anti_only        3

## Stratified Sampling

In [4]:
import os
import re
import json
import requests
import pandas as pd
from query import ANTI, ISRAEL, PALESTINE

a = ANTI
i = ISRAEL
p = PALESTINE

# dedupe is after sampling for fuzzy-identical cases
queries = {
    "ip_not": f'(({i}) OR ({p})) AND NOT ({a})',
    "ip_and": f'(({i}) OR ({p})) AND ({a})',
    "anti_only": f'({a}) AND NOT (({i}) OR ({p}))',
}

# max pages per stratum
PAGINATE_BY_QUERY = {
    "anti_only": 5,
    "ip_not": 3,
    "ip_and": 2,
}

platforms = ["4chan", "8kun", "truthsocial", "bluesky"]

months = [
    ("january", "2025-01-01", "2025-01-31"),
    ("february", "2025-02-01", "2025-02-28"),
    ("march", "2025-03-01", "2025-03-31"),
    ("april", "2025-04-01", "2025-04-30"),
    ("may", "2025-05-01", "2025-05-31"),
    ("june", "2025-06-01", "2025-06-30"),
    ("july", "2025-07-01", "2025-07-31"),
    ("august", "2025-08-01", "2025-08-31"),
    ("september", "2025-09-01", "2025-09-30"),
    ("october", "2025-10-01", "2025-10-31"),
    ("november", "2025-11-01", "2025-11-30"),
    ("december", "2025-12-01", "2025-12-31"),
]

DATA_DIR = "data"
CURSOR_INDEX_PATH = os.path.join(DATA_DIR, "cursor_index.json")

# -----------------------------------------------------------------------------
# SETUP: create data/ and one subfolder per platform if they don't exist
# -----------------------------------------------------------------------------

os.makedirs(DATA_DIR, exist_ok=True)
for platform in platforms:
    os.makedirs(os.path.join(DATA_DIR, platform), exist_ok=True)


# -----------------------------------------------------------------------------
# HELPER: cursor index persistence
# -----------------------------------------------------------------------------

def load_cursor_index():
    if os.path.exists(CURSOR_INDEX_PATH):
        with open(CURSOR_INDEX_PATH) as f:
            return json.load(f)
    return {}

def save_cursor(platform, query_label, month_name, page_num, cursor):
    index = load_cursor_index()
    key = f"{platform}__{query_label}__{month_name}__{page_num}"
    index[key] = cursor
    with open(CURSOR_INDEX_PATH, "w") as f:
        json.dump(index, f, indent=2)

def get_cursor(platform, query_label, month_name, page_num):
    index = load_cursor_index()
    key = f"{platform}__{query_label}__{month_name}__{page_num}"
    return index.get(key)


# -----------------------------------------------------------------------------
# HELPER: coerce mixed-type object columns to string for pyarrow compatibility
# -----------------------------------------------------------------------------

def clean_for_parquet(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str)
    return df


# -----------------------------------------------------------------------------
# HELPER: check if a cell is already saved, recover parquet from csv if needed
# -----------------------------------------------------------------------------

def cell_already_saved(platform, query_label, page_num, month_name):
    base_name = f"{query_label}_{page_num}_{month_name}"
    csv_path = os.path.join(DATA_DIR, platform, f"{base_name}.csv")
    parquet_path = os.path.join(DATA_DIR, platform, f"{base_name}.parquet")
    json_path = os.path.join(DATA_DIR, platform, f"{base_name}.json")

    if os.path.exists(csv_path) and os.path.exists(parquet_path):
        return True

    # csv exists but parquet failed: reconstruct from csv, no re-query needed
    if os.path.exists(csv_path) and not os.path.exists(parquet_path):
        print(f"  recovering parquet from csv: {base_name}")
        try:
            df = pd.read_csv(csv_path)
            clean_for_parquet(df.copy()).to_parquet(parquet_path, engine="pyarrow", index=False)
            if os.path.exists(json_path):
                os.remove(json_path)
            print(f"  parquet recovered successfully")
        except Exception as e:
            print(f"  parquet recovery failed: {e}")
        return True

    # json temp file exists and csv write also failed: reconstruct both
    if os.path.exists(json_path):
        print(f"  recovering from temp json: {base_name}")
        try:
            with open(json_path) as f:
                results = json.load(f)
            df = pd.json_normalize(results)
            if "text" in df.columns:
                df["text"] = df["text"].fillna("").apply(
                    lambda t: re.sub(r'http\S+|www\.\S+', '[URL]', t)
                )
            df.to_csv(csv_path, index=False)
            clean_for_parquet(df.copy()).to_parquet(parquet_path, engine="pyarrow", index=False)
            os.remove(json_path)
            print(f"  both files recovered from temp json")
        except Exception as e:
            print(f"  json recovery failed: {e}")
        return True

    return False


# -----------------------------------------------------------------------------
# HELPER: check if a cell finished early (last saved page had < 10,000 rows)
# or reached its max page cap, by reading the actual saved data rather than
# relying on cursor presence alone. Returns (is_complete, last_page_num).
# -----------------------------------------------------------------------------

def cell_fully_complete(platform, query_label, month_name, max_pages):
    last_page_with_file = 0
    last_file_path = None

    for page_num in range(1, max_pages + 1):
        csv_path = os.path.join(DATA_DIR, platform, f"{query_label}_{page_num}_{month_name}.csv")
        if os.path.exists(csv_path):
            last_page_with_file = page_num
            last_file_path = csv_path
        else:
            break

    if last_page_with_file == 0:
        return False, 0

    last_df = pd.read_csv(last_file_path)
    n_rows_last_page = len(last_df)

    # last saved page had fewer than 10,000 rows: that page triggered
    # early-stop, nothing more to fetch for this cell
    if n_rows_last_page < 10000:
        return True, last_page_with_file

    # last page was full and we've also hit max_pages: cell is complete
    if last_page_with_file >= max_pages:
        return True, last_page_with_file

    # last page was full but more pages remain to fetch
    return False, last_page_with_file


# -----------------------------------------------------------------------------
# HELPER: save one page of results to both csv and parquet
# -----------------------------------------------------------------------------

def save_page(results, platform, query_label, page_num, month_name):
    if not results:
        print(f"  page {page_num}: 0 results, nothing to save")
        return 0

    df = pd.json_normalize(results)

    # strip URLs from text immediately on ingestion
    # [URL] token preserves the signal that a link was present.
    if "text" in df.columns:
        df["text"] = df["text"].fillna("").apply(
            lambda t: re.sub(r'http\S+|www\.\S+', '[URL]', t)
        )

    base_name = f"{query_label}_{page_num}_{month_name}"
    csv_path = os.path.join(DATA_DIR, platform, f"{base_name}.csv")
    parquet_path = os.path.join(DATA_DIR, platform, f"{base_name}.parquet")
    json_path = os.path.join(DATA_DIR, platform, f"{base_name}.json")

    # save csv first
    df.to_csv(csv_path, index=False)

    # temp save raw json in case parquet write fails
    with open(json_path, "w") as f:
        json.dump(results, f)

    try:
        clean_for_parquet(df.copy()).to_parquet(parquet_path, engine="pyarrow", index=False)
        os.remove(json_path)
    except Exception as e:
        print(f"  parquet write failed for {base_name}: {e}")
        print(f"  temp json saved at {json_path} for recovery")

    print(f"  page {page_num}: saved {len(df)} rows -> {csv_path} and {parquet_path}")
    return len(df)


# -----------------------------------------------------------------------------
# MAIN COLLECTION LOOP: month x platform x query x page
# -----------------------------------------------------------------------------

grand_total = 0
skipped = 0

for month_name, since_date, until_date in months:
    print(f"\n########## MONTH: {month_name} ##########")

    for platform in platforms:
        print(f"\n==== PLATFORM: {platform} ====")

        for query_label, term_query in queries.items():
            max_pages = PAGINATE_BY_QUERY[query_label]
            print(f"\n-- query: {query_label} (max {max_pages} pages) --")

            is_complete, last_page = cell_fully_complete(platform, query_label, month_name, max_pages)

            if is_complete:
                print(f"  cell already complete through page {last_page}, skipping entirely")
                skipped += last_page
                continue

            search_after_cursor = get_cursor(platform, query_label, month_name, last_page) if last_page > 0 else None
            page_num = last_page + 1

            while page_num <= max_pages:

                if cell_already_saved(platform, query_label, page_num, month_name):
                    print(f"  page {page_num}: already saved, skipping")
                    skipped += 1
                    search_after_cursor = get_cursor(platform, query_label, month_name, page_num)
                    page_num += 1
                    continue

                params = {
                    "sortdesc": "true",
                    "limit": 10000,
                    "site": platform,
                    "term": term_query,
                    "since": since_date,
                    "until": until_date,
                    "standard_fields": "true",
                    "querytype": "boolean_content",
                }
                if search_after_cursor:
                    params["search_after"] = search_after_cursor

                response = requests.get(f"{API_URL}/content", headers=_headers, params=params)

                if response.status_code != 200:
                    print(f"  page {page_num}: FAILED, status {response.status_code}")
                    print(f"  {response.text[:300]}")
                    break

                data = response.json()
                results = data.get("results", [])
                n_results = len(results)
                total_hits = data.get("total_hits")

                print(f"  page {page_num}: {n_results} results (total_hits: {total_hits})")

                n_saved = save_page(results, platform, query_label, page_num, month_name)
                grand_total += n_saved

                # early stop: a page with fewer than 10,000 results means
                # there's nothing left to paginate through for this cell
                if n_results < 10000:
                    print(f"  page {page_num} returned < 10,000 results, stopping early for this cell")
                    break

                search_after_cursor = data.get("search_after")
                if search_after_cursor:
                    save_cursor(platform, query_label, month_name, page_num, search_after_cursor)
                if not search_after_cursor:
                    print(f"  no further search_after cursor, stopping for this cell")
                    break

                page_num += 1

print(f"\n\nDONE. Total rows collected: {grand_total} new rows. Skipped {skipped} already-saved cells.")


########## MONTH: january ##########

==== PLATFORM: 4chan ====

-- query: ip_not (max 3 pages) --
  cell already complete through page 3, skipping entirely

-- query: ip_and (max 2 pages) --
  cell already complete through page 2, skipping entirely

-- query: anti_only (max 5 pages) --
  cell already complete through page 5, skipping entirely

==== PLATFORM: 8kun ====

-- query: ip_not (max 3 pages) --
  cell already complete through page 1, skipping entirely

-- query: ip_and (max 2 pages) --
  cell already complete through page 1, skipping entirely

-- query: anti_only (max 5 pages) --
  cell already complete through page 1, skipping entirely

==== PLATFORM: truthsocial ====

-- query: ip_not (max 3 pages) --
  cell already complete through page 3, skipping entirely

-- query: ip_and (max 2 pages) --
  cell already complete through page 1, skipping entirely

-- query: anti_only (max 5 pages) --
  cell already complete through page 5, skipping entirely

==== PLATFORM: bluesky ====



C:\Users\jwram\AppData\Local\Temp\ipykernel_9352\89185806.py:157: DtypeWarning: Columns (13,66,67,68,69,70,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  last_df = pd.read_csv(last_file_path)


  cell already complete through page 2, skipping entirely

-- query: anti_only (max 5 pages) --
  cell already complete through page 5, skipping entirely

########## MONTH: march ##########

==== PLATFORM: 4chan ====

-- query: ip_not (max 3 pages) --
  cell already complete through page 3, skipping entirely

-- query: ip_and (max 2 pages) --
  cell already complete through page 2, skipping entirely

-- query: anti_only (max 5 pages) --
  cell already complete through page 5, skipping entirely

==== PLATFORM: 8kun ====

-- query: ip_not (max 3 pages) --
  cell already complete through page 1, skipping entirely

-- query: ip_and (max 2 pages) --
  cell already complete through page 1, skipping entirely

-- query: anti_only (max 5 pages) --
  cell already complete through page 1, skipping entirely

==== PLATFORM: truthsocial ====

-- query: ip_not (max 3 pages) --
  cell already complete through page 3, skipping entirely

-- query: ip_and (max 2 pages) --
  cell already complete through 

## Dedupe and Clean
Exact by ID and Exact by Text Hash
Deduplication is scoped within each platform, so identical cross-platform content (e.g. a post crossposted to multiple platforms) is retained once per platform, preserving platform as a unit of analysis (can be further deduped cross-platform downstream if needed).

In [1]:
import os
import re
import glob
import html
import hashlib
import pandas as pd

DATA_DIR = "data"
platforms = ["4chan", "8kun", "truthsocial", "bluesky"]

# url pattern: catches http/https/ftp/www, plus html-escaped or backslash-escaped
# slashes (e.g. "https:&#47;&#47;..." or "https:\/\/...") that a plain regex misses
URL_RE = re.compile(r'https?:\\?/\\?/\S+|ftp://\S+|www\.\S+\.\S+', re.IGNORECASE)

# 4chan greentext reply-quote header, e.g. ">>514431211>" possibly chained
# (">>514431211>>514431212>"), only stripped from the start of the text
QUOTE_HEADER_RE = re.compile(r'^(?:>>\d+>?\s*)+')

def clean_text(text):
    if not isinstance(text, str):
        return text
    t = html.unescape(text)          # reveal &gt; &#47; etc. before matching
    t = QUOTE_HEADER_RE.sub('', t)   # strip leading 4chan reply-quote chain
    t = URL_RE.sub('', t)            # strip actual urls
    t = re.sub(r'\[URL\]', '', t, flags=re.IGNORECASE)  # strip ingestion-time [URL] token
    return t.strip()

# load
all_dfs = []
for platform in platforms:
    for fpath in glob.glob(os.path.join(DATA_DIR, platform, "*.parquet")):
        df = pd.read_parquet(fpath, engine="pyarrow")
        df["platform"] = platform
        fname = os.path.basename(fpath)
        parts = fname.replace(".parquet", "").split("_")
        df["stratum"] = "_".join(parts[:-1])
        df["source_file"] = fname
        all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)
n_raw = len(combined_df)
print(f"Combined before cleaning: {n_raw} rows across {len(all_dfs)} files")

combined_df.to_parquet(os.path.join(DATA_DIR, "corpus_raw.parquet"), engine="pyarrow", index=False)
print(f"Saved raw corpus: {n_raw} rows")

# ID dedup (order-independent of text cleaning)
combined_df = combined_df.drop_duplicates(subset=["platform", "_source.id"], keep="first")
n_after_id = len(combined_df)
removed_id = n_raw - n_after_id

# clean text (quote headers, urls, [URL] tokens)
combined_df["_source.text"] = combined_df["_source.text"].apply(clean_text)

# drop rows that are now empty after cleaning
is_empty = combined_df["_source.text"].fillna("").str.strip() == ""
removed_empty = int(is_empty.sum())
combined_df = combined_df[~is_empty]
n_after_empty = len(combined_df)

# exact text-hash dedup within platform
combined_df["_hash"] = combined_df["_source.text"].fillna("").apply(
    lambda t: hashlib.sha256(t.strip().lower().encode("utf-8")).hexdigest()
)
combined_df = combined_df.drop_duplicates(subset=["platform", "_hash"], keep="first")
combined_df = combined_df.drop(columns=["_hash"])
n_final = len(combined_df)
removed_hash = n_after_empty - n_final

total_removed = n_raw - n_final

print(f"\n=== Removal breakdown ===")
print(f"Raw rows:                {n_raw}")
print(f"Removed by ID dedup:     {removed_id:>10} ({removed_id/n_raw*100:.2f}% of raw, {removed_id/total_removed*100 if total_removed else 0:.2f}% of removed)")
print(f"Removed by empty text:   {removed_empty:>10} ({removed_empty/n_raw*100:.2f}% of raw, {removed_empty/total_removed*100 if total_removed else 0:.2f}% of removed)")
print(f"Removed by hash dedup:   {removed_hash:>10} ({removed_hash/n_raw*100:.2f}% of raw, {removed_hash/total_removed*100 if total_removed else 0:.2f}% of removed)")
print(f"Total removed:           {total_removed:>10} ({total_removed/n_raw*100:.2f}% of raw)")
print(f"Final rows:               {n_final}")

print(f"\nFinal: {n_final} rows")
print(combined_df.groupby(["platform", "stratum"]).size().reset_index(name="rows").to_string(index=False))

combined_df.to_parquet(os.path.join(DATA_DIR, "corpus_deduped.parquet"), engine="pyarrow", index=False)
print(f"Saved to {os.path.join(DATA_DIR, 'corpus_deduped.parquet')}")

Combined before cleaning: 3446292 rows across 382 files
Saved raw corpus: 3446292 rows

=== Removal breakdown ===
Raw rows:                3446292
Removed by ID dedup:             29 (0.00% of raw, 0.01% of removed)
Removed by empty text:        17615 (0.51% of raw, 3.75% of removed)
Removed by hash dedup:       451479 (13.10% of raw, 96.24% of removed)
Total removed:               469123 (13.61% of raw)
Final rows:               2977169

Final: 2977169 rows
   platform     stratum   rows
      4chan anti_only_1 116249
      4chan anti_only_2 115398
      4chan anti_only_3 114961
      4chan anti_only_4 114265
      4chan anti_only_5 114450
      4chan    ip_and_1 112353
      4chan    ip_and_2 102606
      4chan    ip_not_1 116244
      4chan    ip_not_2 115533
      4chan    ip_not_3 115449
       8kun anti_only_1  23193
       8kun    ip_and_1   7593
       8kun    ip_not_1  32585
    bluesky anti_only_1 114285
    bluesky anti_only_2 114068
    bluesky anti_only_3 113524
    bluesk

In [2]:
# Breakdown of total dedupe removals: identical ID vs. text hash

import hashlib
import pandas as pd

raw_df = pd.read_parquet("data/corpus_raw.parquet", engine="pyarrow")

before = len(raw_df)

after_id = raw_df.drop_duplicates(subset=["platform", "_source.id"], keep="first").copy()
removed_by_id = before - len(after_id)

after_id["_hash"] = after_id["_source.text"].fillna("").apply(
    lambda t: hashlib.sha256(t.strip().lower().encode("utf-8")).hexdigest()
)
after_hash = after_id.drop_duplicates(subset=["platform", "_hash"], keep="first")
removed_by_hash = len(after_id) - len(after_hash)

total_removed = before - len(after_hash)

print(f"Total removed: {total_removed}")
print(f"  Removed by ID dedup:   {removed_by_id} ({removed_by_id / total_removed * 100:.2f}%)")
print(f"  Removed by hash dedup: {removed_by_hash} ({removed_by_hash / total_removed * 100:.2f}%)")

Total removed: 411173
  Removed by ID dedup:   29 (0.01%)
  Removed by hash dedup: 411144 (99.99%)
